In [ ]:
import pandas as pd
import numpy as np
import os
import sys
# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
sys.path.insert(0, src_dir)

# Now you can import the module
import models_utils

# from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor


from joblib import dump, load
import itertools

In [2]:
transcriptomics = load("data/transcriptomics_matched.joblib")
proteomics = load("data/proteomics_matched.joblib")

In [3]:
indices = proteomics.index

## Elastic Net

In [ ]:
elasticnet_model = ElasticNet()

In [ ]:
param_grid_elasticnet = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],  # Regularization strength
    'l1_ratio': [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0],  # Mix of L1 and L2 regularization
    'max_iter': [1000, 5000, 10000],          # Number of iterations
    'tol': [1e-4, 1e-3, 1e-2],                # Tolerance for optimization
    'selection': ['cyclic', 'random'],        # Update rule for coordinate descent
}

In [ ]:
# Use the full adaptive pipeline with ElasticNet
elasticnet_scores = models_utils.full_adaptive_pipeline(
    model=elasticnet_model,
    param_grid=param_grid_elasticnet,
    x=transcriptomics,
    y=proteomics,
    anova_features=20,
    size = 'trial'
)

In [ ]:
elasticnet_scores

## XGBoost

In [32]:
xgb = XGBRegressor(device='cpu', tree_method='hist')

In [ ]:
param_grid_xgboost = {
    'eta': [0.01, 0.03, 0.1, 0.3],         # Learning rate (smaller values for better performance)
    'max_depth': [3, 5, 7, 10],            # Maximum tree depth
    'subsample': [0.5, 0.7, 0.9, 1.0],     # Fraction of samples for training each tree
    'colsample_bytree': [0.5, 0.7, 0.9, 1.0],  # Fraction of features for each tree
    'gamma': [0, 0.1, 0.5, 1],             # Minimum loss reduction to make a further partition
    'min_child_weight': [1, 3, 5, 7],      # Minimum sum of instance weight (Hessian) needed in a child
    'lambda': [0, 1, 5, 10],               # L2 regularization term
    'alpha': [0, 0.1, 0.5, 1],             # L1 regularization term
    'n_estimators': [100, 200, 500],       # Number of boosting rounds
}

In [ ]:
xgb_scores = models_utils.full_adaptive_pipeline(
    model = xgb,
    param_grid = param_grid_xgboost,
    X = transcriptomics,
    y = proteomics,
    anova_features = 20
    size = 'trial'
)

In [16]:
xgb_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,0.095513,-1.005652,"{'eta': 0.05, 'max_depth': 3}"
1,A2M,SH4_SKIN,0.094028,0.226838,"{'eta': 0.05, 'max_depth': 3}"
2,A2M,AU565_BREAST,1.250857,0.147475,"{'eta': 0.05, 'max_depth': 3}"
3,A2M,KMRC1_KIDNEY,-1.680788,-1.203801,"{'eta': 0.05, 'max_depth': 3}"
4,A2M,CAL51_BREAST,0.468455,1.458798,"{'eta': 0.05, 'max_depth': 3}"
